# Learning 01: Zero-Shot Topic Classification with GLiClass

**GLiClass** (Generalist and Lightweight Model for Sequence Classification) is a zero-shot text
classifier that matches the performance of cross-encoders while being significantly faster.
You specify the labels at inference time — no retraining required.

## Architecture

Unlike cross-encoders that encode `[text + label]` pairs and re-encode for every new label,
GLiClass encodes the text once and scores all labels in a **single forward pass**.
This gives near-constant inference time as label count grows.

## Pipeline modes

| `classification_type` | Returns | Use when |
|---|---|---|
| `'multi-label'` | All labels above `threshold` | A text can belong to multiple categories |
| `'single-label'` | The single highest-scoring label | Mutually exclusive categories |

In [1]:
from gliclass import GLiClassModel, ZeroShotClassificationPipeline
from transformers import AutoTokenizer
import torch, json, os

fixtures = os.path.normpath(os.path.join(os.getcwd(), "..", "fixtures", "input"))
with open(os.path.join(fixtures, "news_articles.json")) as f:
    articles = json.load(f)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✓ Device: {device}, loaded {len(articles)} articles")

✓ Device: cuda, loaded 10 articles


## 1. Load Model and Pipeline

In [2]:
model = GLiClassModel.from_pretrained("knowledgator/gliclass-edge-v3.0")
tokenizer = AutoTokenizer.from_pretrained("knowledgator/gliclass-edge-v3.0", add_prefix_space=True)

# Multi-label: a text can belong to multiple topics
pipeline = ZeroShotClassificationPipeline(
    model, tokenizer,
    classification_type='multi-label',
    device=device
)
print("✓ Pipeline ready")

✓ Pipeline ready


## 2. Basic Classification

```python
results = pipeline(text, labels, threshold=0.5)
# → [[{"label": "technology", "score": 0.92}, ...]]
```

The outer list maps to input texts. For a single text, index with `[0]`.

In [3]:
text = articles[0]["text"]  # Apple MacBook article
labels = ["technology", "finance", "sports", "science", "politics", "entertainment"]

results = pipeline(text, labels, threshold=0.3)[0]

print(f"Text: {text[:80]}...\n")
print(f"Results (threshold=0.3):")
for r in sorted(results, key=lambda x: x['score'], reverse=True):
    bar = '█' * int(r['score'] * 20)
    print(f"  {r['label']:15} {bar:20} {r['score']:.3f}")

  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:02<00:00,  2.30s/it]

100%|██████████| 1/1 [00:02<00:00,  2.30s/it]

Text: Apple unveiled the new MacBook Pro featuring the M4 Pro chip with 40% faster CPU...

Results (threshold=0.3):
  technology      █████████████████    0.870


## 3. Threshold Effect

In [4]:
# Multi-topic article: Netflix (entertainment + business)
text = articles[5]["text"]
print(f"Text: {text[:80]}...\n")

for threshold in [0.2, 0.4, 0.6, 0.8]:
    results = pipeline(text, labels, threshold=threshold)[0]
    labels_found = [r['label'] for r in results]
    print(f"  threshold={threshold}: {labels_found}")

Text: Netflix reported a record 22 million new subscribers in Q4, driven by hit series...



  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00, 48.82it/s]

  threshold=0.2: ['technology', 'finance', 'entertainment']


  0%|          | 0/1 [00:00<?, ?it/s]

...

100%|██████████| 1/1 [00:00<00:00, 42.61it/s]

  threshold=0.6: ['technology', 'finance', 'entertainment']


  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00, 43.62it/s]

  threshold=0.8: ['entertainment']


## 4. Batch Processing

In [5]:
texts = [a["text"] for a in articles]
labels = ["technology", "finance", "sports", "science", "politics", "entertainment",
          "world news", "business", "health", "disaster", "space"]

# Process all at once — pipeline batches internally
all_results = pipeline(texts, labels, threshold=0.4, batch_size=4)

print(f"Classified {len(all_results)} articles:\n")
for article, results in zip(articles, all_results):
    top = sorted(results, key=lambda x: x['score'], reverse=True)
    predicted = [r['label'] for r in top[:2]]
    expected = article["expected_topics"]
    match = "✓" if any(p in expected for p in predicted) else "✗"
    print(f"  {match} [{article['domain']:12}] predicted={predicted} expected={expected}")

  0%|          | 0/3 [00:00<?, ?it/s]

 33%|███▎      | 1/3 [00:00<00:00,  2.31it/s]

100%|██████████| 3/3 [00:00<00:00,  6.55it/s]

Classified 10 articles:

  ✓ [tech        ] predicted=['technology'] expected=['technology']
  ✓ [finance     ] predicted=['finance', 'business'] expected=['finance', 'economics']
  ✓ [sports      ] predicted=['sports', 'space'] expected=['sports']
  ✓ [science     ] predicted=['health', 'science'] expected=['science', 'health']
  ✓ [politics    ] predicted=['politics', 'finance'] expected=['politics']
  ✓ [entertainment] predicted=['technology', 'entertainment'] expected=['entertainment', 'business']
  ✓ [world       ] predicted=['disaster'] expected=['disaster', 'world news']
  ✓ [tech        ] predicted=['technology'] expected=['technology', 'business']
  ✓ [space       ] predicted=['space', 'technology'] expected=['science', 'space']
  ✓ [sports      ] predicted=['sports'] expected=['sports']


## 5. Single-Label Classification

When categories are mutually exclusive, use `'single-label'` mode.
The pipeline returns the single highest-scoring label.

In [6]:
pipeline_sl = ZeroShotClassificationPipeline(
    model, tokenizer,
    classification_type='single-label',
    device=device
)

labels_sl = ["technology", "finance", "sports", "science", "politics", "entertainment"]
texts = [a["text"] for a in articles]

results = pipeline_sl(texts, labels_sl, batch_size=4)
print("Single-label classification:")
for article, res in zip(articles, results):
    label = res[0]['label']
    score = res[0]['score']
    expected = article['expected_topics'][0]
    match = "✓" if label == expected or label in article['expected_topics'] else "~"
    print(f"  {match} [{article['domain']:12}] {label:15} ({score:.2f})  expected={expected}")

  0%|          | 0/3 [00:00<?, ?it/s]

100%|██████████| 3/3 [00:00<00:00, 61.30it/s]

Single-label classification:
  ✓ [tech        ] technology      (0.99)  expected=technology
  ✓ [finance     ] finance         (0.97)  expected=finance
  ✓ [sports      ] sports          (0.94)  expected=sports
  ~ [science     ] technology      (0.54)  expected=science
  ~ [politics    ] finance         (0.61)  expected=politics
  ✓ [entertainment] entertainment   (0.68)  expected=entertainment
  ~ [world       ] entertainment   (0.87)  expected=disaster
  ✓ [tech        ] technology      (0.99)  expected=technology
  ~ [space       ] technology      (0.80)  expected=science
  ✓ [sports      ] sports          (0.48)  expected=sports


## 6. Custom Labels

You can use any descriptive English labels — the model is zero-shot.
More specific label descriptions generally improve accuracy.

In [7]:
text = articles[0]["text"]  # Apple MacBook article

# Generic labels
generic = pipeline(text, ["tech", "finance", "sport"], threshold=0.0)[0]
print("Generic labels:",  [(r['label'], round(r['score'],2)) for r in generic])

# Descriptive labels — more context helps
descriptive = pipeline(text, [
    "consumer electronics and software",
    "financial markets and investments",
    "athletic competitions and sports news"
], threshold=0.0)[0]
print("Descriptive labels:", [(r['label'][:30], round(r['score'],2)) for r in descriptive])

  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00, 82.58it/s]

Generic labels: [('tech', 0.76), ('finance', 0.0), ('sport', 0.0)]


  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00, 89.81it/s]

Descriptive labels: [('consumer electronics and softw', 0.96), ('financial markets and investme', 0.07), ('athletic competitions and spor', 0.0)]


## Summary

- `GLiClassModel` + `ZeroShotClassificationPipeline` — standard setup
- `classification_type='multi-label'` — multiple topics per text (use `threshold`)
- `classification_type='single-label'` — one best topic per text
- `pipeline(texts, labels, threshold, batch_size)` — batch processing built-in
- Output: `[[{"label": str, "score": float}, ...]]` — outer list = texts, inner = matched labels
- More descriptive label names → better zero-shot accuracy